# Data Mining - Lab 2
**Adventist University of Central Africa - MSc in Big Data Analytics**

**Dataset:** Credit Approval dataset (crx.data)

Steps done in this notebook:
1. Data Preparation
2. Exploratory Data Analysis (EDA)
3. Preprocessing, Feature Selection and Engineering
4. Model Creation and Evaluation (classification model, neural network, hyperparameter tuning, comparison)

## 1. Data Preparation

### Import libraries

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, f1_score


### Load the dataset

The file `crx.data` has no header row, so we give the columns names A1 to A16.
A16 is the class column: `+` means approved, `-` means not approved.

In [4]:
col_names = ["A1","A2","A3","A4","A5","A6","A7","A8","A9","A10",
             "A11","A12","A13","A14","A15","A16"]

df = pd.read_csv("crx.data", header=None, names=col_names)
df.head()


FileNotFoundError: [Errno 2] No such file or directory: 'crx.data'

### Clean the missing value marker

This dataset uses `?` instead of a blank cell for missing values, so we replace it with `NaN`.
Columns A2 and A14 are actually numbers but were loaded as text because of the `?`, so we convert them back to numeric.

In [ ]:
df = df.replace("?", np.nan)

df["A2"] = pd.to_numeric(df["A2"])
df["A14"] = pd.to_numeric(df["A14"])

df.dtypes


### Save the prepared data to a CSV file (as asked in the lab)

In [ ]:
df.to_csv("credit_approval_prepared.csv", index=False)
print("Saved credit_approval_prepared.csv")


## 2. Exploratory Data Analysis (EDA)

### Check missing values

In [ ]:
df.isnull().sum()


### Summary statistics of the numeric columns

In [ ]:
df.describe()


### Class balance (A16)
We check if the two classes (approved / not approved) are roughly balanced.

In [ ]:
df["A16"].value_counts().plot(kind="bar")
plt.title("Number of applications per class (A16)")
plt.xlabel("Class")
plt.ylabel("Count")
plt.show()


### Histograms of the numeric columns
This helps us see how each numeric variable is spread out.

In [ ]:
df.hist(figsize=(12, 8))
plt.tight_layout()
plt.show()


### Boxplots to spot outliers

In [ ]:
numeric_cols = df.select_dtypes(include="number").columns

plt.figure(figsize=(10, 5))
df[numeric_cols].boxplot()
plt.title("Boxplot of numeric columns (looking for outliers)")
plt.show()


## 3. Preprocessing, Feature Selection and Engineering

### Separate numeric and categorical columns
We keep A16 (the target) out of this list because we treat it separately.

In [ ]:
numeric_cols = df.select_dtypes(include="number").columns.tolist()
categorical_cols = df.select_dtypes(include="object").columns.tolist()
categorical_cols.remove("A16")

print("Numeric columns:", numeric_cols)
print("Categorical columns:", categorical_cols)


### Handle missing values

For numeric columns we fill the missing values with the median, and for categorical
columns we fill them with the most frequent value (mode). This is simpler than dropping
rows and keeps all 690 applications in the dataset.

In [ ]:
for col in numeric_cols:
    df[col] = df[col].fillna(df[col].median())

for col in categorical_cols:
    df[col] = df[col].fillna(df[col].mode()[0])

df.isnull().sum()


### Handle outliers

We use the IQR (interquartile range) method: values that fall far outside the normal
range for a column are capped to the nearest acceptable value instead of being deleted.

In [ ]:
for col in numeric_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    df[col] = df[col].clip(lower, upper)

df[numeric_cols].describe()


### Encode categorical columns

We use Label Encoding to turn each categorical column into numbers, since scikit-learn
models need numeric input. The target column A16 is also label encoded (+ / - becomes 1 / 0).

In [ ]:
encoders = {}

for col in categorical_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
    encoders[col] = le

target_encoder = LabelEncoder()
df["A16"] = target_encoder.fit_transform(df["A16"])

# check what 0 and 1 mean for the target
print(list(target_encoder.classes_))
df.head()


### Scale the numeric columns

We use StandardScaler so that all numeric columns have a similar range. This matters a lot
for the neural network model later, since it is sensitive to the scale of the input features.

In [ ]:
scaler = StandardScaler()
df[numeric_cols] = scaler.fit_transform(df[numeric_cols])

df[numeric_cols].describe()


### Feature selection

We keep all 15 features for now, since the dataset is small (15 features) and dropping
columns without a strong reason could remove useful information. We just double check
that no column is redundant by looking at the correlation between numeric features.

In [ ]:
corr = df[numeric_cols].corr()
plt.figure(figsize=(8, 6))
plt.imshow(corr, cmap="coolwarm")
plt.colorbar()
plt.xticks(range(len(numeric_cols)), numeric_cols, rotation=45)
plt.yticks(range(len(numeric_cols)), numeric_cols)
plt.title("Correlation between numeric features")
plt.show()


### Train / test split

We split the data into 80% training and 20% testing, keeping the class proportions
similar in both sets (`stratify=y`).

In [ ]:
X = df.drop(columns=["A16"])
y = df["A16"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Train size:", X_train.shape, " Test size:", X_test.shape)


## 4. Model Creation and Evaluation

### 4a. Classification model

We start with a Logistic Regression model, since it is a simple and common baseline
for a binary classification problem like this one (approved vs not approved).

In [ ]:
log_model = LogisticRegression(max_iter=1000, random_state=42)
log_model.fit(X_train, y_train)

y_pred_log = log_model.predict(X_test)


**Metrics chosen: Accuracy and F1-score**

- Accuracy tells us the overall percentage of correct predictions, which is easy to understand.
- F1-score combines precision and recall, which is useful here because the two classes
  (approved 44.5% / not approved 55.5%) are not perfectly balanced, so accuracy alone
  could be a bit misleading.

In [ ]:
acc_log = accuracy_score(y_test, y_pred_log)
f1_log = f1_score(y_test, y_pred_log)

print("Logistic Regression -> Accuracy:", round(acc_log, 3), " F1-score:", round(f1_log, 3))


### 4b. Neural network model with 10-fold cross validation

We build a multi-layer neural network (MLPClassifier) with two hidden layers.
We evaluate it using 10-fold cross validation on the training data, then check it
on the test set with the same two metrics as before, so we can compare the models fairly.

In [ ]:
nn_model = MLPClassifier(hidden_layer_sizes=(16, 8), activation="relu",
                          max_iter=1000, random_state=42)

cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

cv_acc = cross_val_score(nn_model, X_train, y_train, cv=cv, scoring="accuracy")
cv_f1 = cross_val_score(nn_model, X_train, y_train, cv=cv, scoring="f1")

print("10-fold CV Accuracy: mean =", round(cv_acc.mean(), 3), " std =", round(cv_acc.std(), 3))
print("10-fold CV F1-score: mean =", round(cv_f1.mean(), 3), " std =", round(cv_f1.std(), 3))


In [ ]:
# fit the neural network on the full training set and check it on the test set
nn_model.fit(X_train, y_train)
y_pred_nn = nn_model.predict(X_test)

acc_nn = accuracy_score(y_test, y_pred_nn)
f1_nn = f1_score(y_test, y_pred_nn)

print("Neural Network -> Accuracy:", round(acc_nn, 3), " F1-score:", round(f1_nn, 3))


### 4c. Hyperparameter tuning with Grid Search

We search over 3 hyperparameters of the neural network:
- `hidden_layer_sizes` (the architecture of the network)
- `activation` (the activation function used in the hidden layers)
- `learning_rate_init` (how big the weight update steps are)

We use Grid Search with 10-fold cross validation, and pick the combination that gives
the best F1-score, since F1-score is the metric we care most about for this imbalanced data.

In [ ]:
param_grid = {
    "hidden_layer_sizes": [(16,), (16, 8), (32, 16)],
    "activation": ["relu", "tanh"],
    "learning_rate_init": [0.001, 0.01]
}

grid_search = GridSearchCV(
    estimator=MLPClassifier(max_iter=1000, random_state=42),
    param_grid=param_grid,
    scoring="f1",
    cv=cv,
    n_jobs=-1
)

grid_search.fit(X_train, y_train)

print("Best parameters:", grid_search.best_params_)
print("Best CV F1-score:", round(grid_search.best_score_, 3))


### 4d. Compare the tuned model against the model in 4a

In [ ]:
best_nn_model = grid_search.best_estimator_
y_pred_tuned = best_nn_model.predict(X_test)

acc_tuned = accuracy_score(y_test, y_pred_tuned)
f1_tuned = f1_score(y_test, y_pred_tuned)

print("Tuned Neural Network -> Accuracy:", round(acc_tuned, 3), " F1-score:", round(f1_tuned, 3))


In [ ]:
# put all three results together to compare them
results = pd.DataFrame({
    "Model": ["Logistic Regression", "Neural Network (default)", "Neural Network (tuned)"],
    "Accuracy": [acc_log, acc_nn, acc_tuned],
    "F1-score": [f1_log, f1_nn, f1_tuned]
})

results


In [ ]:
x = np.arange(len(results))
width = 0.35

plt.figure(figsize=(8, 5))
plt.bar(x - width/2, results["Accuracy"], width, label="Accuracy")
plt.bar(x + width/2, results["F1-score"], width, label="F1-score")
plt.xticks(x, results["Model"], rotation=15)
plt.ylabel("Score")
plt.title("Comparing Logistic Regression vs Neural Network (default and tuned)")
plt.legend()
plt.show()


**Comparison:** The tuned neural network is compared against the Logistic Regression
baseline and the default neural network on both Accuracy and F1-score. Grid Search with
10-fold cross validation helps find hyperparameters that generalize better than a
default, untuned network, and the bar chart above makes the difference between the
three models easy to see at a glance.